# WF2020 Failed-Fold Rally Gap Attribution v1

Purpose: explain the remaining `wf_2020` gap after the return-preserving bull-participation replay. This is a diagnostic notebook, not an actor promotion.

Baselines:
- Canonical: `v19reg_v22_budget_usage_bull_guard_v1`
- Direct comparison: `return_preserving_bull_participation_three_fold_checkpoint_metric_replay_v1`
- Reference: `cap10_three_fold_checkpoint_metric_replay_v2`, `bull_reentry_three_fold_checkpoint_metric_replay_v1`, `recovery_participation_three_fold_checkpoint_metric_replay_v1`

Hypothesis: the remaining calibrated `wf_2020` gap is concentrated in low-exposure rally periods after the drawdown/recovery transition, especially late 2020, rather than in a drawdown-control failure.

Acceptance for this diagnostic run: identify whether a simple post-hoc, non-promotional exposure floor screen can close the remaining `wf_2020` calibrated return floor while keeping MaxDD below 20%, and write the next target/replay action if so.


In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'AGENTS.md').exists() and (candidate / 'runtime').exists():
            return candidate
    raise RuntimeError(f'repo root not found from {start}')

ROOT = find_repo_root(Path.cwd())
EXPERIMENT = 'wf2020_failed_fold_rally_gap_attribution_v1'
CARD = ROOT / 'runtime/model_selection/card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

NEW_REPLAY = ROOT / 'runtime/model_selection/card/return_preserving_bull_participation_three_fold_checkpoint_metric_replay_v1'
CAP10_REPLAY = ROOT / 'runtime/model_selection/card/cap10_three_fold_checkpoint_metric_replay_v2'
BULL_REPLAY = ROOT / 'runtime/model_selection/card/bull_reentry_three_fold_checkpoint_metric_replay_v1'
RECOVERY_REPLAY = ROOT / 'runtime/model_selection/card/recovery_participation_three_fold_checkpoint_metric_replay_v1'
TARGET_CARD = ROOT / 'runtime/model_selection/card/wf2020_return_preserving_bull_participation_target_v1'
FRONTIER_SUMMARY = ROOT / 'runtime/model_selection/card/model_retention_frontier_v1/summary.json'

SIDE_CARS = {
    'candidate_metric_delta': CARD / 'candidate_metric_delta.csv',
    'monthly_path_attribution': CARD / 'monthly_path_attribution.csv',
    'low_exposure_rally_days': CARD / 'low_exposure_rally_days.csv',
    'diagnostic_overlay_screen': CARD / 'diagnostic_overlay_screen.csv',
    'selection': CARD / 'selection.json',
    'summary': CARD / 'summary.json',
    'conclusion': CARD / 'conclusion.md',
}


def rel(path: Path) -> str:
    return str(path.relative_to(ROOT))


def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')

print('card', rel(CARD))


card runtime/model_selection/card/wf2020_failed_fold_rally_gap_attribution_v1


In [2]:
def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))

new_summary = load_json(NEW_REPLAY / 'summary.json')
target_summary = load_json(TARGET_CARD / 'summary.json')
frontier_summary = load_json(FRONTIER_SUMMARY).get('summary', {})
acceptance = target_summary.get('acceptance_criteria', {})
TARGET_RETURN_PCT = float(acceptance.get('wf2020_reference_floor_return_pct', 69.60455156850816))
# The handoff stores the calibrated relative floor directly; return floor is derived from the summary evidence when available.
TARGET_RELATIVE_FLOOR_PP = float(
    target_summary.get('acceptance_criteria', {}).get('wf2020_relative_return_pp_gte', -263.7287958891489)
)
TARGET_MAX_DD_PCT = float(acceptance.get('max_walkforward_drawdown_pct_lt', 20.0))
source_target_summary = target_summary
wf2020_selected = new_summary['selected_by_fold']['wf_2020']
remaining_relative_gap_pp = TARGET_RELATIVE_FLOOR_PP - float(wf2020_selected['selected_relative_return'])
remaining_return_gap_pp = max(0.0, TARGET_RETURN_PCT - float(wf2020_selected.get('selected_return_pct', 57.718671330246885)))
print('target_relative_floor_pp', TARGET_RELATIVE_FLOOR_PP)
print('target_return_pct', TARGET_RETURN_PCT)
print('wf2020_selected_relative', wf2020_selected['selected_relative_return'])
print('remaining_relative_gap_pp', remaining_relative_gap_pp)
print('selected_daily_path', wf2020_selected.get('selected_daily_path'))


target_relative_floor_pp -263.7287958891489
target_return_pct 69.60455156850816
wf2020_selected_relative -275.6146761274102
remaining_relative_gap_pp 11.88588023826128
selected_daily_path runtime/model_selection/card/return_preserving_bull_participation_three_fold_checkpoint_metric_replay_v1/official_daily_paths/wf_2020__bc_initialization_official_daily_path.csv


In [3]:
def load_bc_metrics(card: Path, label: str) -> pd.DataFrame:
    path = card / 'checkpoint_candidate_metrics.csv'
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path)
    df = df[df['fold'].astype(str).eq('wf_2020') & df['candidate'].astype(str).eq('bc_initialization')].copy()
    df['source'] = label
    return df

metrics = pd.concat(
    [
        load_bc_metrics(NEW_REPLAY, 'return_preserving_projected'),
        load_bc_metrics(CAP10_REPLAY, 'cap10_reference'),
        load_bc_metrics(BULL_REPLAY, 'bull_reentry_prior'),
        load_bc_metrics(RECOVERY_REPLAY, 'recovery_unconsumed_target'),
    ],
    ignore_index=True,
)
metric_cols = ['source', 'test_return', 'test_relative_return', 'test_max_dd', 'test_avg_pos']
metrics = metrics[metric_cols].copy()
base = metrics.loc[metrics['source'].eq('return_preserving_projected')].iloc[0]
for col in ['test_return', 'test_relative_return', 'test_max_dd', 'test_avg_pos']:
    metrics[f'{col}_delta_vs_new'] = pd.to_numeric(metrics[col], errors='coerce') - float(base[col])
metrics['gap_to_calibrated_relative_floor_pp'] = TARGET_RELATIVE_FLOOR_PP - pd.to_numeric(metrics['test_relative_return'], errors='coerce')
metrics['gap_to_return_floor_pp'] = TARGET_RETURN_PCT - pd.to_numeric(metrics['test_return'], errors='coerce')
metrics.to_csv(SIDE_CARS['candidate_metric_delta'], index=False)
display(metrics)


,source,test_return,test_relative_return,test_max_dd,test_avg_pos,test_return_delta_vs_new,test_relative_return_delta_vs_new,test_max_dd_delta_vs_new,test_avg_pos_delta_vs_new,gap_to_calibrated_relative_floor_pp,gap_to_return_floor_pp
0,return_preserving_projected,57.718671,-275.614676,17.277968,0.392275,0.000000,0.000000,0.000000,0.000000,11.885880,11.885880
1,cap10_reference,40.580326,-292.753021,14.984701,0.405426,-17.138345,-17.138345,-2.293268,0.013151,29.024226,29.024226
2,bull_reentry_prior,55.597960,-277.735388,15.489488,0.406077,-2.120712,-2.120712,-1.788480,0.013802,14.006592,14.006592
3,recovery_unconsumed_target,28.136997,-305.196351,14.863234,0.378538,-29.581674,-29.581674,-2.414735,-0.013737,41.467555,41.467555


In [4]:
def load_daily(path: Path, label: str) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=['datetime']).sort_values('datetime').reset_index(drop=True)
    df['source'] = label
    df['strategy_ret'] = df['strategy_nav'].pct_change().fillna(0.0)
    df['price_ret'] = df['price'].pct_change().fillna(0.0)
    df['month'] = df['datetime'].dt.to_period('M').astype(str)
    df['mom5_prev'] = df['price'].pct_change(5).shift(1).fillna(0.0)
    prev_nav = df['strategy_nav'].shift(1)
    df['strategy_dd_prev'] = (prev_nav / prev_nav.cummax() - 1.0).fillna(0.0)
    return df

new_daily_path = ROOT / wf2020_selected['selected_daily_path']
cap10_daily_path = CAP10_REPLAY / 'official_daily_paths/wf_2020__bc_initialization_official_daily_path.csv'
new_daily = load_daily(new_daily_path, 'return_preserving_projected')
cap10_daily = load_daily(cap10_daily_path, 'cap10_reference')

def return_pct(nav: pd.Series) -> float:
    return float((nav.iloc[-1] / nav.iloc[0] - 1.0) * 100.0)

def max_drawdown_pct(nav: pd.Series) -> float:
    dd = (nav.cummax() - nav) / nav.cummax()
    return float(dd.max() * 100.0)

monthly_rows = []
for label, df in [('return_preserving_projected', new_daily), ('cap10_reference', cap10_daily)]:
    for month, g in df.groupby('month', sort=True):
        monthly_rows.append({
            'source': label,
            'month': month,
            'rows': int(len(g)),
            'strategy_return_pct': return_pct(g['strategy_nav']),
            'price_return_pct': return_pct(g['price']),
            'avg_position': float(g['position'].mean()),
            'min_position': float(g['position'].min()),
            'max_position': float(g['position'].max()),
            'positive_price_days': int((g['price_ret'] > 0).sum()),
            'low_exposure_positive_days_pos_lt_015': int(((g['price_ret'] > 0) & (g['position'] < 0.15)).sum()),
            'low_exposure_positive_days_pos_lt_030': int(((g['price_ret'] > 0) & (g['position'] < 0.30)).sum()),
        })
monthly = pd.DataFrame(monthly_rows)
monthly.to_csv(SIDE_CARS['monthly_path_attribution'], index=False)
display(monthly[monthly['source'].eq('return_preserving_projected')])


,source,month,rows,strategy_return_pct,price_return_pct,avg_position,min_position,max_position,positive_price_days,low_exposure_positive_days_pos_lt_015,low_exposure_positive_days_pos_lt_030
0,return_preserving_projected,2020-01,21,0.000000,-16.560487,0.077633,0.051962,0.129529,10,10,10
1,return_preserving_projected,2020-02,19,0.000000,-18.943494,0.120898,0.055672,0.162500,11,6,11
2,return_preserving_projected,2020-03,22,-0.011685,-57.949402,0.263015,0.100000,0.667428,10,4,8
3,return_preserving_projected,2020-04,21,5.470425,64.979221,0.364989,0.181808,0.643272,11,0,4
4,return_preserving_projected,2020-05,20,3.080743,38.712863,0.186156,0.176099,0.195942,12,0,12
5,return_preserving_projected,2020-06,22,3.647861,23.659654,0.391720,0.178652,0.603959,15,0,3
6,return_preserving_projected,2020-07,22,8.706474,25.601641,0.225409,0.094439,0.461071,11,3,8
7,return_preserving_projected,2020-08,21,4.049732,11.296308,0.346162,0.099791,0.545625,10,1,6
8,return_preserving_projected,2020-09,21,-2.986357,-10.260865,0.303946,0.095641,0.476641,11,7,7
9,return_preserving_projected,2020-10,22,-1.864109,-7.422562,0.097627,0.097627,0.097627,8,8,8


In [5]:
work = new_daily.copy()
low_exposure_rally = work[
    (work['price_ret'] > 0)
    & (work['position'] < 0.15)
    & (work['datetime'] >= pd.Timestamp('2020-10-01'))
].copy()
low_exposure_rally['missed_floor_030_incremental_ret'] = (0.30 - low_exposure_rally['position']).clip(lower=0.0) * low_exposure_rally['price_ret']
low_exposure_rally['missed_floor_035_incremental_ret'] = (0.35 - low_exposure_rally['position']).clip(lower=0.0) * low_exposure_rally['price_ret']
low_cols = [
    'datetime', 'month', 'price', 'price_ret', 'strategy_nav', 'position', 'action', 'mom5_prev', 'strategy_dd_prev',
    'missed_floor_030_incremental_ret', 'missed_floor_035_incremental_ret',
]
low_exposure_rally[low_cols].to_csv(SIDE_CARS['low_exposure_rally_days'], index=False)
print('low_exposure_rally_days', len(low_exposure_rally))
display(low_exposure_rally[low_cols].head(20))


low_exposure_rally_days 32


,datetime,month,price,price_ret,strategy_nav,position,action,mom5_prev,strategy_dd_prev,missed_floor_030_incremental_ret,missed_floor_035_incremental_ret
189,2020-10-01,2020-10,17.927334,0.062088,1.258991e+06,0.097627,SELL,0.172556,-0.078098,0.012565,0.015669
191,2020-10-05,2020-10,17.923332,0.097616,1.258545e+06,0.097627,HOLD,0.056550,-0.070500,0.019755,0.024636
193,2020-10-07,2020-10,18.680000,0.060319,1.268987e+06,0.097627,HOLD,0.045126,-0.057139,0.012207,0.015223
194,2020-10-08,2020-10,19.328667,0.034725,1.277939e+06,0.097627,HOLD,0.106679,-0.046116,0.007027,0.008764
195,2020-10-09,2020-10,20.342667,0.052461,1.291932e+06,0.097627,HOLD,0.078167,-0.039387,0.010617,0.013240
196,2020-10-12,2020-10,21.226000,0.043423,1.303764e+06,0.097627,SELL,0.245775,-0.028869,0.008788,0.010959
204,2020-10-22,2020-10,19.658001,0.003505,1.294493e+06,0.097627,HOLD,-0.061124,-0.027568,0.000709,0.000885
209,2020-10-29,2020-10,17.445999,0.079624,1.267287e+06,0.097627,HOLD,-0.175095,-0.083566,0.016114,0.020095
211,2020-11-02,2020-11,16.864000,0.016108,1.245520e+06,0.097627,HOLD,-0.143177,-0.071272,0.003260,0.004065
212,2020-11-03,2020-11,17.867332,0.059496,1.283045e+06,0.097627,HOLD,-0.069966,-0.063756,0.012040,0.015015


In [6]:
def simulate_incremental_floor(df: pd.DataFrame, *, floor: float, mom_threshold: float, start: str) -> dict:
    mask = (
        (df['datetime'] >= pd.Timestamp(start))
        & (df['mom5_prev'] >= float(mom_threshold))
        & (df['position'] < float(floor))
    )
    target_position = df['position'].astype(float).copy()
    target_position.loc[mask] = float(floor)
    incremental_ret = (target_position - df['position'].astype(float)) * df['price_ret'].astype(float)
    # Diagnostic approximation: keep the actual replay daily returns and add only the extra price exposure.
    target_ret = df['strategy_ret'].astype(float) + incremental_ret
    nav = (1.0 + target_ret).cumprod() * float(df['strategy_nav'].iloc[0])
    return {
        'floor': float(floor),
        'mom5_prev_threshold': float(mom_threshold),
        'start': start,
        'overlay_days': int(mask.sum()),
        'return_pct': return_pct(nav),
        'return_lift_pp': return_pct(nav) - return_pct(df['strategy_nav']),
        'max_dd_pct': max_drawdown_pct(nav),
        'max_dd_delta_pp': max_drawdown_pct(nav) - max_drawdown_pct(df['strategy_nav']),
        'return_floor_gate': bool(return_pct(nav) >= TARGET_RETURN_PCT),
        'max_dd_gate': bool(max_drawdown_pct(nav) < TARGET_MAX_DD_PCT),
        'diagnostic_gate': bool(return_pct(nav) >= TARGET_RETURN_PCT and max_drawdown_pct(nav) < TARGET_MAX_DD_PCT),
    }

screen_rows = []
for floor in [0.12, 0.15, 0.18, 0.20, 0.22, 0.25, 0.30, 0.35]:
    for mom_threshold in [-0.02, 0.0, 0.02, 0.04, 0.08]:
        for start in ['2020-04-01', '2020-07-01', '2020-10-01', '2020-11-01']:
            screen_rows.append(simulate_incremental_floor(new_daily, floor=floor, mom_threshold=mom_threshold, start=start))
screen = pd.DataFrame(screen_rows)
screen = screen.sort_values(
    ['diagnostic_gate', 'max_dd_pct', 'return_pct', 'overlay_days'],
    ascending=[False, True, False, True],
).reset_index(drop=True)
screen.to_csv(SIDE_CARS['diagnostic_overlay_screen'], index=False)
best_gate = screen[screen['diagnostic_gate']].head(1)
best_row = best_gate.iloc[0].to_dict() if len(best_gate) else {}
print('screen_rows', len(screen), 'diagnostic_pass', int(screen['diagnostic_gate'].sum()))
display(screen.head(20))


screen_rows 160 diagnostic_pass 13


,floor,mom5_prev_threshold,start,overlay_days,return_pct,return_lift_pp,max_dd_pct,max_dd_delta_pp,return_floor_gate,max_dd_gate,diagnostic_gate
0,0.35,-0.02,2020-10-01,44,82.360750,24.642079,17.277968,7.105427e-15,True,True,True
1,0.30,-0.02,2020-10-01,44,77.290851,19.572180,17.277968,7.105427e-15,True,True,True
2,0.35,0.02,2020-10-01,37,76.573510,18.854838,17.277968,7.105427e-15,True,True,True
3,0.35,-0.02,2020-11-01,31,74.341399,16.622727,17.277968,7.105427e-15,True,True,True
4,0.35,0.00,2020-10-01,41,74.045559,16.326888,17.277968,7.105427e-15,True,True,True
5,0.35,0.04,2020-10-01,34,73.604609,15.885937,17.277968,7.105427e-15,True,True,True
6,0.30,0.02,2020-10-01,37,72.751733,15.033061,17.277968,7.105427e-15,True,True,True
7,0.25,-0.02,2020-10-01,44,72.313825,14.595154,17.277968,7.105427e-15,True,True,True
8,0.30,-0.02,2020-11-01,31,70.984334,13.265663,17.277968,7.105427e-15,True,True,True
9,0.30,0.00,2020-10-01,41,70.766980,13.048309,17.277968,7.105427e-15,True,True,True


In [7]:
base_return = return_pct(new_daily['strategy_nav'])
base_max_dd = max_drawdown_pct(new_daily['strategy_nav'])
nov = monthly[(monthly['source'].eq('return_preserving_projected')) & (monthly['month'].eq('2020-11'))].iloc[0].to_dict()
dec = monthly[(monthly['source'].eq('return_preserving_projected')) & (monthly['month'].eq('2020-12'))].iloc[0].to_dict()
low_days = int(len(low_exposure_rally))
low_floor030_incremental_pp = float(low_exposure_rally['missed_floor_030_incremental_ret'].sum() * 100.0)
low_floor035_incremental_pp = float(low_exposure_rally['missed_floor_035_incremental_ret'].sum() * 100.0)

decision = 'wf2020_low_exposure_rally_gap_requires_training_safe_floor_target_no_promotion'
next_mechanism = 'build_training_safe_low_exposure_rally_floor_target_then_rerun_official_three_fold_checkpoint_replay'
if not best_row:
    decision = 'wf2020_low_exposure_rally_gap_unresolved_no_promotion'
    next_mechanism = 'inspect_reward_or_policy_representation_drift_before_more_target_revision'

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'return_preserving_bull_participation_three_fold_checkpoint_metric_replay_v1',
    'reference_baseline': 'soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05',
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'execution_target'],
    'target_window': 'wf_2020',
    'closest_historical_success_case': 'return_preserving_bull_participation target removed target-consumption mismatch and improved wf_2020 relative return by 29.58pp vs the unconsumed recovery replay.',
    'closest_historical_failure_case': 'cap10 and recovery replays kept late-2020 exposure near 10% and missed the calibrated wf_2020 return floor.',
    'why_not_duplicate': 'This notebook attributes the remaining failed-fold gap using official daily paths and a low-exposure rally floor screen instead of changing labels or training.',
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'training_allowed_from_this_card': False,
    'target_relative_floor_pp': TARGET_RELATIVE_FLOOR_PP,
    'target_return_pct': TARGET_RETURN_PCT,
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'base_return_pct': base_return,
    'base_max_dd_pct': base_max_dd,
    'remaining_relative_gap_pp': remaining_relative_gap_pp,
    'remaining_return_gap_pp': max(0.0, TARGET_RETURN_PCT - base_return),
    'november_price_return_pct': float(nov['price_return_pct']),
    'november_strategy_return_pct': float(nov['strategy_return_pct']),
    'november_avg_position': float(nov['avg_position']),
    'december_price_return_pct': float(dec['price_return_pct']),
    'december_strategy_return_pct': float(dec['strategy_return_pct']),
    'december_avg_position': float(dec['avg_position']),
    'low_exposure_rally_days_after_2020_10_01': low_days,
    'low_exposure_floor030_incremental_ret_pp_sum': low_floor030_incremental_pp,
    'low_exposure_floor035_incremental_ret_pp_sum': low_floor035_incremental_pp,
    'diagnostic_overlay_pass_count': int(screen['diagnostic_gate'].sum()),
    'best_diagnostic_overlay': best_row,
    'next_mechanism': next_mechanism,
    'sidecars': [path.name for path in SIDE_CARS.values()],
}
summary = dict(selection)
summary.update({
    'source_replay_summary': rel(NEW_REPLAY / 'summary.json'),
    'source_target_summary': rel(TARGET_CARD / 'summary.json'),
    'frontier_recommended_source_before_this_card': frontier_summary.get('recommended_next_source'),
    'conclusion': [
        f'wf_2020 remains short of the calibrated relative floor by {remaining_relative_gap_pp:.2f}pp while MaxDD is {base_max_dd:.2f}%, below the 20% ceiling.',
        f'November 2020 had price return {float(nov["price_return_pct"]):.2f}% but average position only {float(nov["avg_position"]):.3f}.',
        f'The diagnostic floor overlay screen has {int(screen["diagnostic_gate"].sum())} variants that clear the return floor while keeping MaxDD below 20%.',
        'This is not promotion evidence; it is a next target-design direction.',
    ],
})
write_json(SIDE_CARS['selection'], selection)
write_json(SIDE_CARS['summary'], summary)

best_text = 'none'
if best_row:
    best_text = f"floor={best_row['floor']:.2f}, mom5_prev>={best_row['mom5_prev_threshold']:.2f}, start={best_row['start']}, return={best_row['return_pct']:.2f}%, MaxDD={best_row['max_dd_pct']:.2f}%"
conclusion_lines = [
    f'# {EXPERIMENT} conclusion',
    '',
    f'Decision: `{decision}`.',
    '',
    'Key evidence:',
    f'- Base replay `wf_2020` return `{base_return:.2f}%`, relative `{float(wf2020_selected["selected_relative_return"]):.2f}pp`, MaxDD `{base_max_dd:.2f}%`.',
    f'- Calibrated relative gap remains `{remaining_relative_gap_pp:.2f}pp`; return-floor gap remains `{max(0.0, TARGET_RETURN_PCT - base_return):.2f}pp`.',
    f'- November 2020 SOXL price path return `{float(nov["price_return_pct"]):.2f}%`, but model average position was `{float(nov["avg_position"]):.3f}`.',
    f'- Low-exposure positive rally days after 2020-10-01: `{low_days}`.',
    f'- Diagnostic overlay pass count: `{int(screen["diagnostic_gate"].sum())}`; best overlay: `{best_text}`.',
    '',
    'Risk notes:',
    '- The overlay screen is post-hoc attribution on the official `wf_2020` daily path; it is not a promotion result.',
    '- Any next target must express this as training-safe features/rules and rerun the full three-fold official replay.',
    '',
    f'Next action: `{next_mechanism}`.',
]
SIDE_CARS['conclusion'].write_text('\n'.join(conclusion_lines).rstrip() + '\n', encoding='utf-8')
print(json.dumps(json_safe(summary), indent=2, allow_nan=False)[:6000])


{
  "experiment": "wf2020_failed_fold_rally_gap_attribution_v1",
  "created_at": "2026-06-14T08:18:57.657088+00:00",
  "decision": "wf2020_low_exposure_rally_gap_requires_training_safe_floor_target_no_promotion",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": "return_preserving_bull_participation_three_fold_checkpoint_metric_replay_v1",
  "reference_baseline": "soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05",
  "role": "diagnose",
  "changed_layer": [
    "model_selection",
    "execution_target"
  ],
  "target_window": "wf_2020",
  "closest_historical_success_case": "return_preserving_bull_participation target removed target-consumption mismatch and improved wf_2020 relative return by 29.58pp vs the unconsumed recovery replay.",
  "closest_historical_failure_case": "cap10 and recovery replays kept late-2020 exposure near 10% and missed the calibrated wf_2020 return floor.",
  "why_not_duplicate": "This notebook attributes the re

## Conclusion

This diagnostic confirms whether the remaining `wf_2020` gap is an exposure-timing issue in late rally regimes. It should not be read as promotion evidence; the next valid step is a training-safe target revision followed by a full official three-fold replay.